# AQIS lab: Sprint 1 speech-to-text benchmark (Colab)

Runs one stage of WP-1.1 (`docs/specs/sprints/WP-1.1-S1-asr-benchmark.md`) exactly as the lab commands do on any machine: mount Drive, clone the private repository at the tag `AQIS_TAG`, install the locked environments, run `python -m aqis_lab.bench <action>`, write the unit log, and free the runtime.

**Before you start (once, about 15 minutes):**
- Hugging Face: create a **Read** token. Open `ai4bharat/Svarah` and `CohereLabs/cohere-transcribe-03-2026` (and any other gated page the prep names) while signed in and accept their terms.
- Colab Secrets (key icon): add `HF_TOKEN` with notebook access on. `GITHUB_TOKEN` is already there from phase 0.
- Drive: `MyDrive/AQIS_LAB` needs at least 10 GB free.

**Each session:** edit only cell 1 (`STAGE`, `MODELS`, `DTYPE`), pick the runtime type below, then **Run all**. The last cell writes the unit log and frees the runtime, so the GPU stops costing units.

| STAGE | Runtime | What it does |
| --- | --- | --- |
| `prep` | CPU | downloads, selects, cleans, renders the 7 conditions |
| `smoke` | **L4 GPU** | every model on 3 clips per slice; prints hypotheses beside references |
| `anchor` | **L4 GPU** | the 500-clip anchor for Parakeet TDT v2, Canary-Qwen and Cohere, then the anchor check |
| `pilot` | **L4 GPU** | 100 utterances per slice in clean, phone and webrtc |
| `full` | **L4 GPU** | everything else (rerun the same stage after a dropped session: finished files are skipped) |
| `speed` | **L4 GPU** | batch-1 latency, 50 clips |
| `score`, `report` | CPU | number tables, then scorecard, weakness map and `report/s1_report.zip` |

Use an L4. If none is offered, run only the NeMo models on a T4 (set `DTYPE = "fp16"`), or ask before using an A100.

In [ ]:
# 1) Settings: edit STAGE, MODELS and DTYPE only. Then Google Drive is mounted.
AQIS_TAG = "s1-colab-1"
STAGE = "prep"          # prep | smoke | anchor | pilot | full | speed | score | report
MODELS = ["whisper-large-v3", "whisper-large-v3-turbo", "parakeet-tdt-0.6b-v2", "parakeet-unified-0.6b",
          "canary-qwen-2.5b", "granite-speech-4.1-2b", "cohere-transcribe-2b", "qwen3-asr-1.7b", "voxtral-realtime-4b"]
DTYPE = None            # None keeps each model's own dtype (bf16); "fp16" for a NeMo model on a T4

REPO = "https://github.com/NasirAman-promcare/voice-bot-vad-harness.git"
WORKDIR = "/content/aqis"
SCRATCH = "/content/s1_work"
DRIVE_MOUNT = "/content/drive"
AQIS_DRIVE = "/content/drive/MyDrive/AQIS_LAB"
S1_OUT = AQIS_DRIVE + "/s1"

STAGES = ("prep", "smoke", "anchor", "pilot", "full", "speed", "score", "report")
assert STAGE in STAGES, f"STAGE must be one of {STAGES}"

from google.colab import drive

drive.mount(DRIVE_MOUNT)

In [ ]:
# 2) Clone the private repository at AQIS_TAG. The GITHUB_TOKEN secret is read inside this one git command's
#    environment; it is never stored in a variable, a URL, .git/config or any output.
import base64
import os
import subprocess

from google.colab import userdata

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", AQIS_TAG, REPO, WORKDIR],
               env=dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                        GIT_CONFIG_VALUE_0="Authorization: Basic " + base64.b64encode(
                            ("x-access-token:" + userdata.get("GITHUB_TOKEN")).encode()).decode()),
               check=True)

In [ ]:
# 3) Environments. The lab's own .venv (hash-checked phase 0 lock) is always built: it runs prep, score, report and the
#    unit log. prep and report also get the s1-prep lock on top. A GPU stage builds one venv per model stack, each from
#    its own hash-pinned lock, with the phase 0 lock as the build constraint for the two source-only packages.
#    A failed install is printed and recorded in SETUP_ERROR, so the last cell still frees the runtime.
import re

import yaml

GPU_STAGES = ("smoke", "anchor", "pilot", "full", "speed")
PY = WORKDIR + "/.venv/bin/python"
LOCKS = WORKDIR + "/requirements/aqis_lab"
STACK_PY = {}
SETUP_ERROR = None
try:
    subprocess.run(["pip", "install", "--quiet", "uv==0.8.17"], check=True)
    subprocess.run(["bash", "aqis_lab/deploy/dev_bootstrap.sh"], cwd=WORKDIR, check=True)
    if STAGE in ("prep", "report"):
        subprocess.run(["uv", "pip", "install", "--python", PY, "--require-hashes", "--build-constraints",
                        LOCKS + "/phase0.txt", "-r", LOCKS + "/s1-prep.txt"], check=True)
    cfg = yaml.safe_load(open(WORKDIR + "/config/aqis_lab/s1.yaml"))
    unknown = [m for m in MODELS if m not in cfg["models"]]
    assert not unknown, f"not in s1.yaml: {unknown}"
    if STAGE == "anchor":
        MODELS = [m for m in MODELS if m in cfg["stages"]["anchor"]["models"]]
    if STAGE in GPU_STAGES:
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, check=True).stdout.strip())
        probe = open(WORKDIR + "/docs/environment/colab_probe.txt").read()
        colab_python = re.search(r"Python Version:\s*(\d+\.\d+)", probe).group(1)
        for stack in sorted({cfg["models"][m]["stack"] for m in MODELS}):
            venv = f"/content/venv-{stack}"
            subprocess.run(["uv", "venv", "--clear", "--python", colab_python, venv], check=True)
            subprocess.run(["uv", "pip", "sync", "--python", venv + "/bin/python", "--require-hashes",
                            "--build-constraints", LOCKS + "/phase0.txt", f"{LOCKS}/s1-{stack}.txt"], check=True)
            STACK_PY[stack] = venv + "/bin/python"
            print("ready:", stack, venv)
except (subprocess.CalledProcessError, AssertionError) as exc:
    SETUP_ERROR = repr(exc)
    print("SETUP FAILED:", SETUP_ERROR)

In [ ]:
# 4) Unit log start (the runtime is billed from here), then the stage. HF_TOKEN goes into each subprocess's environment
#    only. A failing model is recorded and the next one still runs, so the last cell always frees the runtime.
env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"), PYTHONUNBUFFERED="1")
common = ["--out", S1_OUT, "--work-dir", SCRATCH]
dtype_args = ["--dtype", DTYPE] if DTYPE else []
results = {}


def bench(python, action, *extra):
    return subprocess.run([python, "-m", "aqis_lab.bench", action, *common, *extra], cwd=WORKDIR, env=env).returncode


if SETUP_ERROR:
    print("Skipping the stage because setup failed:", SETUP_ERROR)
else:
    subprocess.run([PY, "-m", "aqis_lab", "unitlog", "start", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
    if STAGE in ("prep", "score", "report"):
        results[STAGE] = bench(PY, STAGE)
    else:
        for model in MODELS:
            python = STACK_PY[cfg["models"][model]["stack"]]
            print("=" * 20, STAGE, model)
            if STAGE == "smoke":
                results[model] = bench(python, "smoke", "--model", model, *dtype_args)
            elif STAGE == "speed":
                results[model] = bench(python, "speed", "--model", model, *dtype_args)
            else:
                results[model] = bench(python, "run", "--stage", STAGE, "--model", model, *dtype_args)
        if STAGE == "anchor":
            results["score"] = bench(PY, "score")
            anchor_json = S1_OUT + "/report/anchor.json"
            print(open(anchor_json).read() if os.path.exists(anchor_json) else "no anchor.json")
    print("exit codes (0 = done; smoke gives 1 when a model returned empty or identical text; 2 = error):")
    for name, rc in results.items():
        print(f"  {name:28s} {rc}")

In [ ]:
# 5) End of session: append this session's estimated compute units to AQIS_LAB/logs/units.csv, then free the runtime so
#    the GPU stops costing units. Read the exit codes above before the next session.
try:
    subprocess.run([PY, "-m", "aqis_lab", "unitlog", "end", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
except subprocess.CalledProcessError as exc:
    print("UNIT LOG NOT WRITTEN, note this session's hours by hand:", exc)
from google.colab import runtime

runtime.unassign()